In [ ]:
import os
print(os.getpid())

In [ ]:
from cobra.io import load_json_model
from cobra.flux_analysis import pfba

import time
import cProfile
import pstats

In [ ]:
# Contains the main steps of the BayesOpt
%run BayesOpt_MOBO_comprehensive.ipynb

In [ ]:
# Plotting functions to be used across notebooks
%run Plotting_MOBO_comprehensive.ipynb

In [ ]:
# load iBsu1103 producing surfactin (objective function: biomass + surfactin production, 95:5 ratio)
model_iBsu1103 = load_json_model("..//Bsubtilis//iBsu1103_producing_surfactin_combined_objective.json")
medium_M9_iBsu1103 = model_iBsu1103.medium
print(model_iBsu1103)
print(model_iBsu1103.objective)

In [ ]:
"""
Media, bounds and costs for iBsu1103 model (different IDs than iJO1366 and iML1515)
"""
medium_M9_iBsu1103 = {
    'EX_cpd00013_b': 9.3745, # Ammonia
    'EX_cpd00063_b': 0.05, # Calcium
    'EX_cpd00027_b': 10, # Carbon - Glucose
    'EX_cpd00254_b': 1, # Magnesium
    'EX_cpd10516_b': 0.1, # Iron Fe3+ (reactant in biomass reaction)
    'EX_cpd00009_b': 34.90, # Phosphate 
    'EX_cpd00205_b': 11.02, # Potassium
    'EX_cpd00048_b': 1, # Sulfate
    'EX_cpd00011_b': 0.0, # Carbon Dioxide 
    'EX_cpd00067_b': 0.0, # Hydrogen 
    'EX_cpd00001_b': 0.0, # Water 
    'EX_cpd00007_b': 20.0 # Oxygen 
}

bounds_M9_iBsu1103 = {
    'EX_cpd00013_b': (0.0, 10),
    'EX_cpd00063_b': (0.0, 10),
    'EX_cpd00027_b': (1.0, 10),
    'EX_cpd00254_b': (0.0, 10),
    'EX_cpd10516_b': (0.0, 10),
    'EX_cpd00009_b': (0.0, 50),
    'EX_cpd00205_b': (0.0, 20),
    'EX_cpd00048_b': (0.0, 10),
    'EX_cpd00011_b': (0.0, 10),  
    'EX_cpd00067_b': (0.0, 10),  
    'EX_cpd00001_b': (0.0, 10),  
    'EX_cpd00007_b': (0, 20) 
} # 12 variable components

costs_M9_iBsu1103 = {
    'EX_cpd00013_b': 10.099587, 
    'EX_cpd00063_b': 18.08223, 
    'EX_cpd00027_b': 7.7647236, 
    'EX_cpd00254_b': 19.1022, 
    'EX_cpd10516_b': 0.1, 
    'EX_cpd00009_b': 23.4234, 
    'EX_cpd00205_b': 20.82177, 
    'EX_cpd00048_b': 19.1022, 
    'EX_cpd00011_b': 0.0, 
    'EX_cpd00067_b': 0.0, 
    'EX_cpd00001_b': 0.0, 
    'EX_cpd00007_b': 0.0, 
}

In [ ]:
# set n_iter and date to be used in all calls and names
date = "2026-03-09"
n_start = 10 # how many random media compositions to initialise the algorithm
n_iter = 5 # how many media compositions to evaluate
run = "1st"
iterations = str(n_iter)
n_candidates = 5 # batch size

medium = medium_M9_iBsu1103
bounds = bounds_M9_iBsu1103
costs = costs_M9_iBsu1103

biomass_rxn_id = "bio00006"
protein_rxn_id = "surfactin_D_production"
opt_objective = "growth-production"


In [ ]:
results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
)

In [ ]:
start_time = time.time() # when did the algorithm start

profiler1 = cProfile.Profile()
profiler1.enable()

results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler1.disable()
profiler1.dump_stats("profile_iBsu1103_gp_1.prof")

 print the 30 most expensive functions
stats1 = pstats.Stats(profiler1)
stats1.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_1 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_1)

plot_pareto_batch_colour(
    results_iBsu1103_gp, 
    "growth rate tensors", 
    "production tensors", 
    figname = (basename_1 + "_growth-production_pareto.png"),
    MetModel = model_iBsu1103,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = model_iBsu1103.objective
)

In [ ]:
'''
SECOND RUN
'''
#basename_1 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")
n_start = None # how many random media compositions to initialise the algorithm
run = "2nd"

start_time = time.time() # when did the algorithm start

profiler2 = cProfile.Profile()
profiler2.enable()

results_iBsu1103_gp = media_BayesOpt(
        MetModel = model_iBsu1103,
        medium = medium,
        bounds = bounds,
        costs = costs,
        opt_objective = opt_objective,
        biomass_objective = biomass_rxn_id,
        production_objective = protein_rxn_id,
        n_start = n_start, 
        data_start = ("Results\\" + basename_1 + ".json"),
        n_iter = n_iter,
        n_candidates = n_candidates,
        model_objective = model_iBsu1103.objective,
        start_time = start_time,
        medium_linear_equality_constraints = None,
        medium_linear_inequality_constraints = None,
        medium_nonlinear_inequality_constraints = None,
        use_pfba = False
    )

profiler2.disable()
profiler2.dump_stats("profile_iBsu1103_gp_2.prof")

# print the 30 most expensive functions
stats2 = pstats.Stats(profiler2)
stats2.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_2 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_2)


In [ ]:
'''
THIRD RUN
'''

n_start = None # how many random media compositions to initialise the algorithm
run = "3rd"

start_time = time.time() # when did the algorithm start

profiler3 = cProfile.Profile()
profiler3.enable()

results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    data_start = ("Results\\" + basename_2 + ".json"),
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
)

profiler3.disable()
profiler3.dump_stats("profile_iBsu1103_gp_3.prof")

# print the 30 most expensive functions
stats3 = pstats.Stats(profiler3)
stats3.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_3 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_3)


In [ ]:
n_start = None # how many random media compositions to initialise the algorithm
run = "4th"

start_time = time.time() # when did the algorithm start

profiler4 = cProfile.Profile()
profiler4.enable()

results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    data_start = ("Results\\" + basename_3 + ".json"),
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
)

profiler4.disable()
profiler4.dump_stats("profile_iBsu1103_gp_4.prof")

# print the 30 most expensive functions
stats4 = pstats.Stats(profiler4)
stats4.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_4 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_4)


In [ ]:
'''
FIFTH RUN
'''

n_start = None # how many random media compositions to initialise the algorithm
run = "5th"

start_time = time.time() # when did the algorithm start

profiler5 = cProfile.Profile()
profiler5.enable()

results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    data_start = ("Results\\" + basename_4 + ".json"),
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
)

profiler5.disable()
profiler5.dump_stats("profile_iBsu1103_gp_5.prof")

# print the 30 most expensive functions
stats5 = pstats.Stats(profiler5)
stats5.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_5 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_5)


In [ ]:
'''
SIXTH RUN
'''
n_start = None # how many random media compositions to initialise the algorithm
run = "6th"

start_time = time.time() # when did the algorithm start

profiler6 = cProfile.Profile()
profiler6.enable()

results_iBsu1103_gp = media_BayesOpt(
    MetModel = model_iBsu1103,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    data_start = ("Results\\" + basename_5 + ".json"),
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = model_iBsu1103.objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
)

profiler6.disable()
profiler6.dump_stats("profile_iBsu1103_gp_6.prof")

# print the 30 most expensive functions
stats6 = pstats.Stats(profiler6)
stats6.sort_stats("cumtime").print_stats(30)

# plot & save results
basename_6 = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_" + run + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iBsu1103_gp, basename_6)


In [ ]:
# load results from all iterations
iter1_file = ("Results\\" + basename_1 + ".json")
iter2_file = ("Results\\" + basename_2 + ".json")
iter3_file = ("Results\\" + basename_3 + ".json")
iter4_file = ("Results\\" + basename_4 + ".json")
iter5_file = ("Results\\" + basename_5 + ".json")
iter6_file = ("Results\\" + basename_6 + ".json")

iteration_files = [iter1_file, iter2_file, iter3_file, iter4_file, iter5_file, iter6_file]

# create a dictionary to store combined results (consider parameters, only non-pareto points from first 5 iterations)
iBsu1103_results_combined = {
    "medium list" : [],
    "is pareto" :torch.tensor([], dtype=torch.bool),
    "growth rate tensors" : torch.tensor([], dtype=torch.double),
    "production tensors" : torch.tensor([], dtype=torch.double),
    "cost tensors" : torch.tensor([], dtype=torch.double),
    "biomass objective" : "",
    "production objective" : "",
    "model objective" : None,
    "n_start" : 0,
    "n_iter" : 0,
    "n_candidates" : 0
}

# extract non pareto points from each iteration and combine them (from 1st to 5th iteration)
for file in iteration_files:
    results = JSON_deserialize_load_results((file), model_iBsu1103)
    is_pareto = results["is pareto"]
    medium_list = results["medium list"]
    growth_tensors = results["growth rate tensors"]
    production_tensors = results["production tensors"]
    cost_tensors = results["cost tensors"]
    # filter non-pareto points (all pareto points will be added later from sixth iteration file)
    non_pareto_medium_list = [m for m, pareto in zip(medium_list, is_pareto) if not pareto]
    non_pareto_growth_tensors = growth_tensors[~torch.tensor(is_pareto)]
    non_pareto_production_tensors = production_tensors[~torch.tensor(is_pareto)]
    non_pareto_cost_tensors = cost_tensors[~torch.tensor(is_pareto)]

    # append to combined results
    iBsu1103_results_combined["medium list"].extend(non_pareto_medium_list)
    iBsu1103_results_combined["growth rate tensors"] = torch.cat((
        iBsu1103_results_combined["growth rate tensors"], 
        non_pareto_growth_tensors), dim = 0)
    iBsu1103_results_combined["production tensors"] = torch.cat((
        iBsu1103_results_combined["production tensors"], 
        non_pareto_production_tensors), dim = 0)
    iBsu1103_results_combined["cost tensors"] = torch.cat((
        iBsu1103_results_combined["cost tensors"], 
        non_pareto_cost_tensors), dim = 0)
    iBsu1103_results_combined["is pareto"]= torch.cat((
        iBsu1103_results_combined["is pareto"], 
    torch.tensor([False]*len(non_pareto_medium_list), dtype=torch.bool)), dim = 0)
    # update other info
    iBsu1103_results_combined["biomass objective"] = results["biomass objective"]
    iBsu1103_results_combined["production objective"] = results["production objective"]
    iBsu1103_results_combined["model objective"] = results["model objective"]
    # just store the first random data for intial points
    if file == iteration_files[0]: iBsu1103_results_combined["n_start"] = results["n_start"]
    # sum the number of iterations across runs
    iBsu1103_results_combined["n_iter"] += results["n_iter"]
    iBsu1103_results_combined["n_candidates"] = results["n_candidates"]

# extract all points from sixth iteration and add them to combined results
results = JSON_deserialize_load_results((iter6_file), model_iBsu1103)
iter6_medium_list = results["medium list"]
iter6_growth_tensors = results["growth rate tensors"]
iter6_production_tensors = results["production tensors"]
iter6_cost_tensors = results["cost tensors"]
is_pareto_iter6 = results["is pareto"]

# add sixth iteration medium list
iBsu1103_results_combined["medium list"].extend(iter6_medium_list)
# add sixth iteration growth rates
iBsu1103_results_combined["growth rate tensors"] = torch.cat((
iBsu1103_results_combined["growth rate tensors"], 
iter6_growth_tensors), dim = 0)
# add sixth iteration production rates
iBsu1103_results_combined["production tensors"] = torch.cat((
iBsu1103_results_combined["production tensors"], 
iter6_production_tensors), dim = 0)
# add sixth iteration costs
iBsu1103_results_combined["cost tensors"] = torch.cat((
iBsu1103_results_combined["cost tensors"], 
iter6_cost_tensors), dim = 0)
# add sixth iteration is pareto
iBsu1103_results_combined["is pareto"] = torch.cat((
iBsu1103_results_combined["is pareto"], 
torch.tensor(is_pareto_iter6, dtype=torch.bool)), dim = 0)
# update other info
iBsu1103_results_combined["biomass objective"] = results["biomass objective"]
iBsu1103_results_combined["production objective"] = results["production objective"]
iBsu1103_results_combined["model objective"] = results["model objective"]
iBsu1103_results_combined["n_iter"] += results["n_iter"]
iBsu1103_results_combined["n_candidates"] = results["n_candidates"]

# save results to JSON file
basename_combined = (date + "_BayesOpt_iBsu1103_" + opt_objective + "_combined_6runs_each" + iterations + "iter")
JSON_serialize_store_results(iBsu1103_results_combined, basename_combined)

In [ ]:
"""Plot all intermediate results"""
"""
# load results from all iterations
iter1_file = ("Results\\" + basename_1 + ".json")
iter2_file = ("Results\\" + basename_2 + ".json")
iter3_file = ("Results\\" + basename_3 + ".json")
iter4_file = ("Results\\" + basename_4 + ".json")
iter5_file = ("Results\\" + basename_5 + ".json")
iter6_file = ("Results\\" + basename_6 + ".json")

iteration_files = [iter1_file, iter2_file, iter3_file, iter4_file, iter5_file]
"""
# extract non pareto points from each iteration and combine them (from 1st to 5th iteration)
for file in iteration_files:
    results = JSON_deserialize_load_results((file), model_iBsu1103)
    print(file[8:-5])
    plot_pareto_batch_colour(
        results, 
        "growth rate tensors", 
        "production tensors", 
        figname = (file[8:-5] + ".png"),
        MetModel = model_iBsu1103,
        initial_medium = medium,
        initial_costs = costs,
        model_objective = model_iBsu1103.objective
    )
    